# Datos, modelos y registro de corridas
Cada muestra se abre al solicitarla. El módulo de entrenamiento solo recibe TRAIN normal y VAL. Las transformaciones geométricas se aplican conjuntamente a imagen y máscara.

In [ ]:
import platform
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone
from importlib.metadata import distributions, version
import numpy as np
import pandas as pd
import torch
from PIL import Image
from lightning.pytorch import LightningDataModule, Trainer, seed_everything
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms.functional import pil_to_tensor
from anomalib.data import ImageBatch
from anomalib.models import Dinomaly, Draem, EfficientAd, Patchcore

MODELS = {
    "patchcore": Patchcore,
    "efficientad-s": EfficientAd,
    "draem-ref": Draem,
    "dinomaly": Dinomaly,
}

## Particiones públicas y privadas

In [ ]:
def public_rows():
    base = DATA / "benchmarks" / "mvtec-ad" / "bottle"
    normal = sorted((base / "train" / "good").glob("*.png"))
    if len(normal) < 12:
        raise ValueError(f"Instala la categoría bottle de MVTec AD en {base}.")
    rows = []
    for i, path in enumerate(normal):
        rows.append((path, None, 0, "valid" if i % 10 == 0 else "train"))
    for path in sorted((base / "test").glob("*/*.png")):
        defect = path.parent.name != "good"
        mask = (
            base / "ground_truth" / path.parent.name / (path.stem + "_mask.png")
            if defect
            else None
        )
        rows.append((path, mask, int(defect), "test"))
    frame = pd.DataFrame(rows, columns=["path", "mask", "label", "split"])
    if set(frame.loc[frame.split == "test", "label"]) != {0, 1}:
        raise ValueError("Faltan imágenes de prueba de MVTec AD.")
    return frame


def cap_rows():
    base = dataset_location()
    frozen = base / "frozen"
    manifest = frozen / "manifest.csv"
    source = json.loads((frozen / "source.json").read_text())
    if digest(manifest) != source["manifest"]:
        raise ValueError("El manifiesto congelado cambió.")
    frame = pd.read_csv(manifest)
    frame["path"] = frame.file.map(lambda name: base / "raw" / name)
    frame["mask"] = frame["mask"].map(lambda name: frozen / name)
    frame["label"] = frame.label.map({"normal": 0, "defect": 1})
    for row in frame.itertuples():
        if digest(row.path) != row.image_hash or digest(row.mask) != row.mask_hash:
            raise ValueError(f"Imagen o máscara modificada: {row.file}")
    return frame, digest(frozen / "source.json")

## Lectura de imágenes y máscaras bajo demanda

In [ ]:
class Samples(Dataset):
    def __init__(self, frame, transform):
        self.rows = list(frame.itertuples())
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row.path) as image:
            pixels = tv_tensors.Image(pil_to_tensor(image.convert("RGB")).float() / 255)
        if row.mask is None:
            mask = torch.zeros(pixels.shape[-2:], dtype=torch.bool)
        else:
            with Image.open(row.mask) as image:
                mask = pil_to_tensor(image.convert("L"))[0] > 0
        pixels, mask = self.transform(pixels, tv_tensors.Mask(mask))
        return pixels, mask, row.label, str(row.path)


def collate(samples):
    images, masks, labels, paths = zip(*samples)
    return ImageBatch(
        image=torch.stack(images),
        gt_mask=torch.stack(masks),
        gt_label=torch.tensor(labels),
        image_path=list(paths),
    )


class Partitions(LightningDataModule):
    def __init__(self, frame, transform, batch):
        super().__init__()
        self.train_batch_size = batch
        self.training = Samples(
            frame[(frame.split == "train") & (frame.label == 0)], transform
        )
        self.validation = Samples(frame[frame.split == "valid"], transform)

    def train_dataloader(self):
        return DataLoader(
            self.training,
            batch_size=self.train_batch_size,
            shuffle=True,
            collate_fn=collate,
            num_workers=0,
        )

    def val_dataloader(self):
        return DataLoader(
            self.validation, batch_size=1, collate_fn=collate, num_workers=0
        )

## Construcción de las cuatro arquitecturas

In [ ]:
def build_model(cfg, variant=None):
    cls = MODELS[cfg["model"]]
    args = {"image_size": (cfg["resize"], cfg["resize"])}
    if cfg["crop"]:
        args["crop_size"] = cfg["crop"]
    transform = cls.configure_pre_processor(**args).transform
    params = dict(cfg["params"])
    if cls is Draem:
        params["dtd_dir"] = DATA / "assets" / "dtd"
    if cls is EfficientAd:
        params["imagenet_dir"] = DATA / "assets" / "imagenette"
    if variant:
        cls = PairedDraem
    model = cls(
        **params,
        pre_processor=False,
        post_processor=False,
        evaluator=False,
        visualizer=False,
    )
    return model, transform

## Huellas y evidencia de etapas previas

In [ ]:
def fingerprint(cfg):
    return hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()


def implementation_hash():
    sources = sorted((ROOT / "notebooks" / "shared").glob("*.ipynb"))
    return fingerprint([(p.relative_to(ROOT).as_posix(), digest(p)) for p in sources])


def evidence(stage, cfg):
    for path in RUNS.glob("*/record.json"):
        record = json.loads(path.read_text(encoding="utf-8"))
        if (
            record["stage"] == stage
            and record["status"] == "PASS"
            and record["implementation_hash"] == implementation_hash()
            and torch.cuda.is_available()
            and record["gpu"] == torch.cuda.get_device_name()
            and record["config_hash"] == fingerprint(cfg)
            and record["lock_hash"] == digest(ROOT / "uv.lock")
        ):
            return True
    return False

## Ejecución y registro

In [ ]:
def fit_run(cfg, stage, seed, variant=None, normal_fraction=1.0, flip=False):
    if (
        stage not in ("preflight", "benchmark", "baseline", "synthetic", "ablation")
        or seed not in cfg["seeds"]
    ):
        raise ValueError("Revisa la etapa y la semilla.")
    if stage != "preflight" and not evidence("preflight", cfg):
        raise ValueError("Falta el preflight aprobado para esta configuración.")
    if stage in ("baseline", "synthetic", "ablation") and not evidence(
        "benchmark", cfg
    ):
        raise ValueError(
            "Completa primero la evaluación pública de esta configuración."
        )
    run = RUNS / (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-"
        + uuid.uuid4().hex[:8]
    )
    run.mkdir(parents=True)
    record = dict(
        stage=stage,
        seed=seed,
        config=cfg,
        config_hash=fingerprint(cfg),
        status="RUNNING",
        lock_hash=digest(ROOT / "uv.lock"),
        python=sys.version,
        platform=platform.platform(),
        packages={
            p: version(p)
            for p in ("torch", "torchvision", "anomalib", "lightning", "roboflow")
        },
        cuda=torch.version.cuda,
    )
    record["implementation_hash"] = implementation_hash()
    (run / "packages.txt").write_text(
        "\n".join(
            sorted(f"{d.metadata['Name']}=={d.version}" for d in distributions())
        ),
        encoding="utf-8",
    )
    save_json(run / "record.json", record)
    revision = subprocess.run(["git","rev-parse","HEAD"], cwd=ROOT, capture_output=True, text=True)
    record["git_commit"] = revision.stdout.strip() or None
    changes = subprocess.run(["git","status","--porcelain","--","notebooks","configs","pyproject.toml","uv.lock"],
                             cwd=ROOT, capture_output=True, text=True)
    record["code_modified"] = bool(changes.stdout.strip()) if changes.returncode == 0 else None
    start = time.perf_counter()
    try:
        if not torch.cuda.is_available():
            raise RuntimeError("PyTorch no detecta una GPU CUDA.")
        seed_everything(seed, workers=True)
        torch.cuda.reset_peak_memory_stats()
        record["gpu"] = torch.cuda.get_device_name()
        record["driver"] = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
            text=True,
        ).strip()
        if stage in ("baseline", "synthetic", "ablation"):
            frame, record["dataset_hash"] = cap_rows()
            record["dataset"] = dataset_location().relative_to(DATA).as_posix()
        else:
            frame = public_rows()
            record["dataset"] = "mvtec-ad/bottle"
            record["dataset_hash"] = fingerprint(
                [
                    (p.relative_to(DATA).as_posix(), digest(p))
                    for p in list(frame.path) + list(frame["mask"].dropna())
                ]
            )
        if stage == "preflight":
            frame = frame.groupby(["split", "label"], sort=False).head(8)
        os.chdir(ROOT)
        model, transform = build_model(cfg, variant)
        if stage == "synthetic":
            if (
                not evidence("baseline", cfg)
                or cfg["model"] != "draem-ref"
                or variant not in ("draem-project", "draem-mixed")
            ):
                raise ValueError("La variante necesita una referencia DRAEM completa.")
            model.pairs, record["pair_hash"], _ = synthetic_pairs(frame)
            model.variant = record["variant"] = variant
            model.pair_transform = transform
        if stage == "ablation":
            if (
                not evidence("baseline", cfg)
                or normal_fraction not in (0.1, 0.25, 0.5, 1.0)
                or (flip and normal_fraction != 1.0)
            ):
                raise ValueError(
                    "Conserva una sola intervención y completa primero el baseline."
                )
            normals = frame[(frame.split == "train") & (frame.label == 0)]
            ordered = normals.sample(frac=1, random_state=seed)
            chosen = ordered.head(max(1, round(len(ordered) * normal_fraction)))
            frame = pd.concat([chosen, frame[frame.split != "train"]])
            record.update(
                normal_fraction=normal_fraction,
                horizontal_flip=flip,
                train_files=chosen.file.tolist(),
            )
        module = Partitions(frame, transform, cfg["train_batch_size"])
        if flip and stage == "ablation":
            from torchvision.transforms.v2 import Compose, RandomHorizontalFlip

            module.training.transform = Compose(
                [RandomHorizontalFlip(p=0.5), transform]
            )
        settings = dict(
            devices=1,
            num_sanity_val_steps=0,
            max_steps=cfg["max_steps"],
            max_epochs=cfg["max_epochs"],
        )
        if stage == "preflight":
            settings.update(max_steps=-1, max_epochs=1)
        settings.update(model.trainer_arguments)
        trainer = Trainer(
            accelerator="gpu",
            precision=cfg["precision"],
            default_root_dir=run,
            logger=False,
            enable_checkpointing=False,
            **settings,
        )
        start_fit = time.perf_counter()
        trainer.fit(model, datamodule=module)
        if cfg["model"] == "efficientad-s":
            trainer.validate(model, datamodule=module, verbose=False)
        torch.cuda.synchronize()
        record["fit_seconds"] = time.perf_counter() - start_fit
        record["steps"] = trainer.global_step
        model.eval().cuda()
        sample = next(iter(module.val_dataloader())).image.cuda()
        with (
            torch.inference_mode(),
            torch.autocast("cuda", enabled=cfg["precision"] == "16-mixed"),
        ):
            start_eval = time.perf_counter()
            output = model(sample)
            torch.cuda.synchronize()
            record["inference_seconds"] = time.perf_counter() - start_eval
            if (
                not torch.isfinite(output.pred_score).all()
                or not torch.isfinite(output.anomaly_map).all()
            ):
                raise ValueError("La inferencia produjo puntuaciones no finitas.")
        if cfg["model"] == "patchcore":
            record["bank_bytes"] = model.model.memory_bank.nbytes
        if stage == "preflight" and torch.cuda.max_memory_reserved() / 2**30 > 9.5:
            raise MemoryError("El pico reservado supera el margen de 9,5 GiB.")
        torch.save(model.state_dict(), run / "weights.pt")
        if stage != "preflight":
            evaluate_validation(model, transform, frame, cfg, run, stage)
        record["status"] = "PASS"
    except Exception as error:
        record.update(status="FAIL", cause=str(error), error_type=type(error).__name__)
        raise
    finally:
        record["seconds"] = time.perf_counter() - start
        if torch.cuda.is_available():
            record["peak_reserved_gib"] = torch.cuda.max_memory_reserved() / 2**30
            record["peak_allocated_gib"] = torch.cuda.max_memory_allocated() / 2**30
        save_json(run / "record.json", record)
        export_run(run, record)
        print("Registro de la corrida:", run)
    return run